In [ ]:
# Install only missing libraries without re-downloading heavy core tools
!pip install streamlit plotly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 63.5 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
import streamlit as st

# Download the missing NLTK stopwords dataset
nltk.download('stopwords')

# Test NLTK download
stop_words = set(stopwords.words('english'))
print(f"✅ Setup verified! Total English stop words loaded: {len(stop_words)}")

✅ Setup verified! Total English stop words loaded: 198


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Download additional required NLTK resources
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')

# 1. Create Sample Product Review Dataset (You can replace this with pd.read_csv('your_dataset.csv'))
data = {
    'review': [
        "This product is amazing! Excellent quality and super fast delivery.",
        "Worst purchase ever. The item arrived broken and customer support is terrible.",
        "Average quality. It works fine, but nothing special.",
        "Extremely happy with this buy! Highly recommended to everyone.",
        "Completely useless product. Stopped working after two days.",
        "Decent product for the price. Works as expected.",
        "Fantastic experience! The build quality is top notch.",
        "Horrible service and delayed shipping. Very disappointed.",
        "Satisfactory purchase, value for money.",
        "Waste of money! Poor quality and defective item."
    ],
    'sentiment': ['Positive', 'Negative', 'Neutral', 'Positive', 'Negative', 'Neutral', 'Positive', 'Negative', 'Positive', 'Negative']
}

df = pd.DataFrame(data)

# 2. Text Preprocessing Function
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()  # Convert to lowercase
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # Remove punctuation and numbers
    words = text.split()
    cleaned_words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    return " ".join(cleaned_words)

# Apply preprocessing
df['cleaned_review'] = df['review'].apply(clean_text)

print("✅ Dataset loaded & preprocessed successfully!")
df[['review', 'cleaned_review', 'sentiment']].head()


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


✅ Dataset loaded & preprocessed successfully!


,review,cleaned_review,sentiment
0,This product is amazing! Excellent quality and...,product amazing excellent quality super fast d...,Positive
1,Worst purchase ever. The item arrived broken a...,worst purchase ever item arrived broken custom...,Negative
2,"Average quality. It works fine, but nothing sp...",average quality work fine nothing special,Neutral
3,Extremely happy with this buy! Highly recommen...,extremely happy buy highly recommended everyone,Positive
4,Completely useless product. Stopped working af...,completely useless product stopped working two...,Negative


In [ ]:
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# 1. Train-Test Split
X = df['cleaned_review']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 2. TF-IDF Vectorization
vectorizer = TfidfVectorizer()
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# 3. Model Training
model = LogisticRegression()
model.fit(X_train_tfidf, y_train)

# 4. Model Evaluation
y_pred = model.predict(X_test_tfidf)
print("🎯 Model Accuracy:", accuracy_score(y_test, y_pred))
print("\n📊 Classification Report:\n", classification_report(y_test, y_pred))

# 5. Save Model and Vectorizer Artifacts for Deployment
joblib.dump(model, 'sentiment_model.pkl')
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')

print("✅ Model & Vectorizer saved as 'sentiment_model.pkl' and 'tfidf_vectorizer.pkl'")

🎯 Model Accuracy: 0.3333333333333333

📊 Classification Report:
               precision    recall  f1-score   support

    Negative       0.33      1.00      0.50         1
     Neutral       0.00      0.00      0.00         1
    Positive       0.00      0.00      0.00         1

    accuracy                           0.33         3
   macro avg       0.11      0.33      0.17         3
weighted avg       0.11      0.33      0.17         3

✅ Model & Vectorizer saved as 'sentiment_model.pkl' and 'tfidf_vectorizer.pkl'


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
%%writefile app.py
import streamlit as st
import joblib
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Setup NLTK
nltk.download('stopwords')
nltk.download('wordnet')

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    words = text.split()
    return " ".join([lemmatizer.lemmatize(w) for w in words if w not in stop_words])

# Load Model Artifacts
model = joblib.load('sentiment_model.pkl')
vectorizer = joblib.load('tfidf_vectorizer.pkl')

st.title("🛒 Product Review Sentiment Analyzer")
st.write("Enter a customer review below to analyze its sentiment.")

user_review = st.text_area("Customer Review:", "")

if st.button("Analyze Sentiment"):
    if user_review.strip() != "":
        cleaned = clean_text(user_review)
        transformed = vectorizer.transform([cleaned])
        prediction = model.predict(transformed)[0]

        if prediction == 'Positive':
            st.success(f"Result: 😊 **{prediction} Sentiment**")
        elif prediction == 'Negative':
            st.error(f"Result: 😡 **{prediction} Sentiment**")
        else:
            st.warning(f"Result: 😐 **{prediction} Sentiment**")
    else:
        st.info("Please enter some text to analyze.")

Writing app.py


In [ ]:
!pip install pyngrok

In [ ]:
from pyngrok import ngrok
import os
import time

# Terminate existing ngrok tunnels if any
ngrok.kill()

# Set up port for Streamlit
port = 8501

# Start Streamlit in the background
os.system(f"streamlit run app.py &")

# Wait 3 seconds for Streamlit to start
time.sleep(3)

# Open a public HTTP tunnel using ngrok
public_url = ngrok.connect(port)
print("🚀 Your Streamlit App is LIVE at:")
print(public_url)

ERROR:pyngrok.process.ngrok:t=2026-10-08T18:43:43+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-10-08T18:43:43+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"


PyngrokNgrokError: The ngrok process errored on start: authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n.

In [ ]:
from google.colab import output
import subprocess

# Start Streamlit on port 8501 in the background
subprocess.Popen(["streamlit", "run", "app.py"])

# Expose port 8501 directly through Colab
output.serve_kernel_port_as_iframe(8501, height='600')

<IPython.core.display.Javascript object>

In [ ]:
import subprocess
import time

# 1. Start Streamlit in the background
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501"])

# 2. Give Streamlit 5 seconds to load fully
print("⏳ Waiting for Streamlit server to start...")
time.sleep(5)

# 3. Print Endpoint IP
!curl -s https://ipv4.icanhazip.com

# 4. Start localtunnel
!npx localtunnel --port 8501

⏳ Waiting for Streamlit server to start...
35.254.236.245
⠙⠹⠸⠼⠴⠦⠧⠇your url is: https://long-bananas-fail.loca.lt
^C
